# Hito 4 — Entrenamiento del Modelo Predictivo de Éxito de Autopartes

**Proyecto:** AutoParts Pro / AutoRepuestos PRO
**Algoritmo:** Random Forest Classifier exportado a ONNX
**Objetivo:** Predecir el nivel de rotación y riesgo comercial de un nuevo repuesto antes de importarlo.

---

### 1. Instalación de dependencias y librerías

In [ ]:
!pip install -q scikit-learn skl2onnx onnx onnxruntime pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

print("✅ Librerías importadas correctamente.")

### 2. Carga y exploración del dataset
*(Asegúrate de subir el archivo `dataset_entrenamiento.csv` en la pestaña de archivos a la izquierda)*

In [ ]:
df = pd.read_csv('dataset_entrenamiento.csv')

print(f"📊 Total de registros cargados: {len(df)}")
print("\nDistribución de las Clases Objetivo:")
print(df['etiqueta_exito'].value_counts())

df[['producto', 'marca', 'modelo', 'categoria_repuesto', 'tipo_desgaste', 'segmento_parque', 'costo', 'precio1', 'ventas_netas', 'etiqueta_exito']].head()

### 3. Definición de variables y Pipeline de preprocesamiento

In [ ]:
features_num = ['costo', 'precio1', 'margen_relativo', 'ratio_precio_mayor']
features_cat = ['categoria_repuesto', 'tipo_desgaste', 'segmento_parque']

X = df[features_num + features_cat]
y = df['clase_exito'] # 2: Alta Rotación, 1: Media, 0: Riesgo/Lenta

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)

print("✅ Pipeline de preprocesamiento configurado.")

### 4. Entrenamiento y evaluación del modelo (Random Forest)

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        class_weight='balanced',
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("====================================================")
print("📈 REPORTE DE EVALUACIÓN DEL MODELO")
print("====================================================")
print(f"Precisión global (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=['Riesgo (0)', 'Media (1)', 'Alta Rotación (2)']))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Riesgo', 'Media', 'Alta'], 
            yticklabels=['Riesgo', 'Media', 'Alta'])
plt.title('Matriz de Confusión - Random Forest')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

### 5. Importancia de Variables (Feature Importances)

In [ ]:
classifier = model_pipeline.named_steps['classifier']
onehot_cols = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(features_cat)
all_features = features_num + list(onehot_cols)

importances = pd.Series(classifier.feature_importances_, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.head(10).plot(kind='barh', color='teal')
plt.title('Top 10 Variables Más Influyentes en la Predicción')
plt.gca().invert_yaxis()
plt.xlabel('Importancia Relativa')
plt.show()

### 6. Exportación del modelo a formato ONNX

In [ ]:
# Entrenamos con el 100% de los datos para la versión final
model_pipeline.fit(X, y)

initial_type = [
    ('costo', FloatTensorType([None, 1])),
    ('precio1', FloatTensorType([None, 1])),
    ('margen_relativo', FloatTensorType([None, 1])),
    ('ratio_precio_mayor', FloatTensorType([None, 1])),
    ('categoria_repuesto', StringTensorType([None, 1])),
    ('tipo_desgaste', StringTensorType([None, 1])),
    ('segmento_parque', StringTensorType([None, 1]))
]

onnx_model = convert_sklearn(model_pipeline, initial_types=initial_type, target_opset=12)

onnx_filename = "modelo_exito_autopartes.onnx"
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"✅ Modelo exportado exitosamente a: {onnx_filename}")

# Probar sesión ONNX
sess = rt.InferenceSession(onnx_filename)
print("🔍 Entradas esperadas por ONNX:", [inp.name for inp in sess.get_inputs()])
print("🔍 Salidas generadas por ONNX:", [out.name for out in sess.get_outputs()])

### 7. Descargar el modelo y metadata para el backend NestJS

In [ ]:
from google.colab import files

# 1. Descargar modelo ONNX
files.download("modelo_exito_autopartes.onnx")

# 2. Guardar y descargar metadata de categorías
metadata = {
    "categorias_validas": list(df['categoria_repuesto'].unique()),
    "tipos_desgaste": list(df['tipo_desgaste'].unique()),
    "segmentos_parque": list(df['segmento_parque'].unique()),
    "clases": {
        "0": "Baja Rotación / Riesgo",
        "1": "Rotación Media",
        "2": "Alta Rotación"
    }
}

with open("modelo_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

files.download("modelo_metadata.json")
print("📥 Descargas iniciadas. Recuerda colocarlos en backend/src/models/")